<a href="https://colab.research.google.com/github/BDH-teacher/RL_from_basics/blob/main/RL_from_basic_ch_9_ActorCriticipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TD Actor-Critic 구현

## 라이브러리 import 및 하이퍼 파라미터 정의

In [ ]:
# 최신버전으로 수정

!pip install gym pyvirtualdisplay > /dev/null 2>&1
!pip install gymnasium[classic-control] > /dev/null 2>&1

In [ ]:
import base64
import collections
import glob
import io
import random

import gymnasium as gym # 최신버전으로 수정
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

from IPython import display as ipythondisplay
from IPython.display import HTML

In [ ]:
# Hyperparameters

learning_rate = 0.0002
gamma         = 0.98
n_rollout     = 10

- 밸류 네트워크가 평가를 대신해 주기 때문에 액터-크리틱 기반 방법론은 학습할 때 리턴을 필요로 하지 않음 <br/><br/>
- 리턴을 관측할 때 까지 기다릴 필요없이 바로 업데이트를 할 수 있고, n개의 데이터를 모아서 업데이트도 가능함
   - n_rollout을 추가하여 몇 번의 상태 전이를 모아서 업데이트할 지 정함

## 액터 크리틱 클래스

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self):
        super(ActorCritic, self).__init__()
        self.data = []

        self.fc1 = nn.Linear(4,128)
        self.fc_pi = nn.Linear(128,2)
        self.fc_v = nn.Linear(128,1)
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def pi(self, x, softmax_dim = 0):
        x = F.relu(self.fc1(x))
        x = self.fc_pi(x)
        prob = F.softmax(x, dim=softmax_dim)
        return prob

    def v(self, x):
        x = F.relu(self.fc1(x))
        v = self.fc_v(x)
        return v

    def put_data(self, transition):
        self.data.append(transition)

    def make_batch(self):
        s_lst, a_lst, r_lst, s_prime_lst, done_lst = [], [], [], [], []
        for transition in self.data:
            s,a,r,s_prime,done = transition
            s_lst.append(s)
            a_lst.append([a])
            r_lst.append([r/100.0])
            s_prime_lst.append(s_prime)
            done_mask = 0.0 if done else 1.0
            done_lst.append([done_mask])

        s_batch, a_batch, r_batch, s_prime_batch, done_batch = torch.tensor(s_lst, dtype=torch.float), torch.tensor(a_lst), \
                                                               torch.tensor(r_lst, dtype=torch.float), torch.tensor(s_prime_lst, dtype=torch.float), \
                                                               torch.tensor(done_lst, dtype=torch.float)
        self.data = []
        return s_batch, a_batch, r_batch, s_prime_batch, done_batch

    def train_net(self):
        s, a, r, s_prime, done = self.make_batch()
        td_target = r + gamma * self.v(s_prime) * done
        delta = td_target - self.v(s)

        pi = self.pi(s, softmax_dim=1)
        pi_a = pi.gather(1,a)
        loss = -torch.log(pi_a) * delta.detach() + F.smooth_l1_loss(self.v(s), td_target.detach())

        self.optimizer.zero_grad()
        loss.mean().backward()
        self.optimizer.step()

- 정책 네트워크 : pi 함수
- 밸류 네트워크 : v 함수
   - 두 뉴럴넷은 하나의 레이어를 공유함
   - 따라서 맨 아래 H1 레이어는 정책 네트워크를 업데이트할 때도, 밸류 네트워크를 업데이트할 때도 항상 업데이트됨 <br/><br/>
- make_batch 함수 : n_rollout동안 모였던 데이터를 s끼리, a끼리, r끼리, s'끼리, done끼리 따로따로 모아서 미니 배치를 만들어 주는 함수 <br/><br/>
- loss는 정책 네트워크의 손실 함수와 밸류 네트워크의 손실 함수를 더하여 한 번에 업데이트를 진행하는 방식 <br/><br/>
- delta에 detach()가 붙어있는 이유는 delta를 상수 취급하기 위함임
   - 해당 값을 계산하기까지 필요했던 모든 그래프의 연산들을 **떼어내기(detach)** 때문
   - 그래야 백 프로파게이션 단계에서 그라디언트가 뒤로 흘러가지 않게됨
   - 만일 detach를 해주지 않는다면, 업데이트시 delta의 값을 계산하는데 쓰였던 밸류 네트워크의 파라미터들 또한 함께 변해버림 <br/><br/>
- 밸류 네트워크의 loss는 TD 방식을 이용해 계산함 <br/><br/>
- 총 n_rollout만큼의 데이터에 대해 각각의 데이터마다 loss가 구해지는데, 이 n개의 loss의 평균을 최종 loss로 정함 -> 이후 backward 함수를 호출하여 그라디언트 계산

## 메인 함수

In [ ]:
def main():
    env = gym.make('CartPole-v1')
    model = ActorCritic()
    print_interval = 20
    score = 0.0

    max_score = 0.0

    for n_epi in range(5000):
        done = False
        s, info = env.reset()

        while not done:
            for t in range(n_rollout):
                prob = model.pi(torch.from_numpy(s).float())
                m = Categorical(prob)
                a = m.sample().item()
                s_prime, r, terminated, truncated, info = env.step(a)

                model.put_data((s,a,r,s_prime,terminated))

                s = s_prime
                score += r

                if terminated or truncated:
                    done = True
                    break

            model.train_net()

        if n_epi%print_interval==0 and n_epi!=0:
            print("# of episode :{}, avg score : {:.1f}".format(n_epi, score/print_interval))

            if score > max_score:
                print(f'>>>> save actor-critic.pth: {score:.1f}')
                torch.save(model.state_dict(), 'actor-critic.pth')
                max_score = score

            score = 0.0
    env.close()

In [ ]:
main()

/tmp/ipython-input-451/1145085741.py:36: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  s_batch, a_batch, r_batch, s_prime_batch, done_batch = torch.tensor(s_lst, dtype=torch.float), torch.tensor(a_lst), \


# of episode :20, avg score : 20.6
>>>> save actor-critic.pth: 412.0
# of episode :40, avg score : 17.5
# of episode :60, avg score : 19.9
# of episode :80, avg score : 21.9
>>>> save actor-critic.pth: 438.0
# of episode :100, avg score : 25.6
>>>> save actor-critic.pth: 512.0
# of episode :120, avg score : 21.6
# of episode :140, avg score : 20.9
# of episode :160, avg score : 22.0
# of episode :180, avg score : 24.8
# of episode :200, avg score : 24.4
# of episode :220, avg score : 30.4
>>>> save actor-critic.pth: 608.0
# of episode :240, avg score : 31.9
>>>> save actor-critic.pth: 637.0
# of episode :260, avg score : 27.2
# of episode :280, avg score : 33.8
>>>> save actor-critic.pth: 675.0
# of episode :300, avg score : 37.3
>>>> save actor-critic.pth: 746.0
# of episode :320, avg score : 44.6
>>>> save actor-critic.pth: 893.0
# of episode :340, avg score : 35.1
# of episode :360, avg score : 47.1
>>>> save actor-critic.pth: 942.0
# of episode :380, avg score : 39.6
# of episode :

- 정책 함수인 model.pi를 통해 액션을 선택하고, 상태전이와 보상을 관찰하여 값들을 model.put에 저장해둠 <br/><br/>
- 이후, n_rollout 만큼 반복 후 model.train_net을 호출하여 학습을 진행함

## 실행

In [ ]:
#import shutil
#import os

env = gym.make('CartPole-v1', render_mode='rgb_array')
model = ActorCritic()
model.load_state_dict(torch.load('actor-critic.pth'))

# Remove existing video directory to ensure a fresh recording
#if os.path.exists('./video_actor_critic'):
#    shutil.rmtree('./video_actor_critic')

# Recreate the environment wrapper for recording. Set episode_trigger to record every episode.
env = gym.wrappers.RecordVideo(env, './video_actor_critic')

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


In [ ]:
s, info = env.reset()
done = False

while not done:
    prob = model.pi(torch.from_numpy(s).float())
    m = Categorical(prob)
    action = m.sample()
    s_prime, r, terminated, truncated, info = env.step(action.item())
    done = terminated or truncated
    s = s_prime
    print(action.item(), r)
env.close()

0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
1 1.0
0 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
0 1.0
1 1.0
0 1.0
1 1.0
1 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.0
0 1.0
1 1.0
0 1.

In [ ]:
# play recorded video
def show_video():
    mp4list = glob.glob('video_actor_critic/*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        ipythondisplay.display(HTML(data='''
            <video alt="test" autoplay loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
            </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

In [ ]:
show_video()